# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SymbolPamnani/Flyrank-ML-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Answer

The feature vector uses information that can reasonably exist before making the ranking decision.

I use keyword/context and content-property features:

- `search_volume`
- `competition`
- `cpc`
- `word_count`
- `char_count`
- `content_age_days`
- `days_since_last_update`
- `competition_level`
- `content_type`
- `main_intent`
- `age_tier`
- `freshness_tier`
- `word_count_tier`
- `char_count_tier`

Numeric missing values are filled with the median of the available data, while categorical missing values are filled with `"unknown"`.

I also add explicit missingness indicators for numeric fields so that missing data is not silently treated as a real zero. This follows the dataset guidance because missingness is systematic across content types.

The feature vector excludes identifiers, trend-derived fields, and search-performance outcome fields that overlap with the trend window.

In [5]:
import pandas as pd
import numpy as np

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

# Features that are reasonable candidates before the prediction/ranking decision
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

# Work on a copy
feature_df = df[numeric_features + categorical_features].copy()

# Add missingness indicators before filling values
for col in numeric_features:
    feature_df[f"{col}_missing"] = feature_df[col].isna().astype(int)

# Numeric imputation: median
for col in numeric_features:
    feature_df[col] = feature_df[col].fillna(feature_df[col].median())

# Categorical imputation
for col in categorical_features:
    feature_df[col] = feature_df[col].fillna("unknown").astype(str)

# One-hot encode categorical variables
X = pd.get_dummies(
    feature_df,
    columns=categorical_features,
    dtype=int
)

print("Original rows:", len(df))
print("Feature matrix shape:", X.shape)
print("\nNumeric source features:", len(numeric_features))
print("Categorical source features:", len(categorical_features))
print("Final vector dimensions:", X.shape[1])

print("\nMissing values remaining in feature matrix:")
print(X.isna().sum().sum())

Original rows: 30000
Feature matrix shape: (30000, 44)

Numeric source features: 7
Categorical source features: 7
Final vector dimensions: 44

Missing values remaining in feature matrix:
0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Answer

| Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| `search_volume` | Estimated search volume for the target keyword | Median + missing flag | Yes, as keyword metadata |
| `competition` | Keyword competition score | Median + missing flag | Yes, as keyword metadata |
| `cpc` | Estimated cost per click | Median + missing flag | Yes, as keyword metadata |
| `word_count` | Number of words in the content | Median + missing flag | Yes, from content |
| `char_count` | Number of characters in the content | Median + missing flag | Yes, from content |
| `content_age_days` | Days since content creation | Median + missing flag | Yes, can be calculated at prediction time |
| `days_since_last_update` | Days since the content was updated | Median + missing flag | Yes, can be calculated at prediction time |
| `competition_level` | Categorical competition bucket | `"unknown"` | Yes |
| `content_type` | Type of content | `"unknown"` | Yes |
| `main_intent` | Search intent category | `"unknown"` | Yes when available |
| `age_tier` | Content-age bucket | `"unknown"` | Yes |
| `freshness_tier` | Update-freshness bucket | `"unknown"` | Yes |
| `word_count_tier` | Word-count bucket | `"unknown"` | Yes |
| `char_count_tier` | Character-count bucket | `"unknown"` | Yes |

The starter dataset does not provide exact historical timestamps for when each metadata field became available, so availability is treated as a practical feature-timing assumption rather than a verified historical fact.

In [6]:
print("Feature availability and missingness check:\n")

for col in numeric_features + categorical_features:
    missing = df[col].isna().sum()
    missing_pct = missing / len(df) * 100

    print(
        f"{col:25s} | "
        f"missing: {missing:5d} ({missing_pct:5.1f}%)"
    )

print("\nFinal feature matrix:")
print("Rows:", X.shape[0])
print("Columns:", X.shape[1])
print("Missing values:", X.isna().sum().sum())

Feature availability and missingness check:

search_volume             | missing:  2468 (  8.2%)
competition               | missing:  2468 (  8.2%)
cpc                       | missing:  2468 (  8.2%)
word_count                | missing:  7699 ( 25.7%)
char_count                | missing:  7699 ( 25.7%)
content_age_days          | missing:     0 (  0.0%)
days_since_last_update    | missing:     0 (  0.0%)
competition_level         | missing:  2610 (  8.7%)
content_type              | missing:     0 (  0.0%)
main_intent               | missing:  2374 (  7.9%)
age_tier                  | missing:     0 (  0.0%)
freshness_tier            | missing:     0 (  0.0%)
word_count_tier           | missing:  7699 ( 25.7%)
char_count_tier           | missing:  7699 ( 25.7%)

Final feature matrix:
Rows: 30000
Columns: 44
Missing values: 0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## Answer

I checked three main leakage risks.

**1. Label-derived fields:** `trend_direction` defines the decline label and `trend_pct` is used to calculate that trend. Neither is included in the feature vector.

**2. Future or overlapping windows:** the last-30-day and previous-30-day fields directly define `trend_direction`. The 90-day performance fields also cover periods that overlap those trend windows. I therefore excluded these performance fields from the feature vector.

**3. Product/decision-derived fields:** fields such as `impression_tier` and `position_tier` summarize search-performance outcomes and are not necessary for this metadata-only feature vector. IDs such as `content_id` and `client_id` are also excluded because they are identifiers, not predictive features.

As a deliberate leakage test, I use `trend_pct` to reconstruct the decline label. Because the label is defined by `trend_direction`, a threshold of less than −20% should reproduce the decline label almost perfectly. This demonstrates why `trend_pct` must not be used as a model feature.

In [7]:
# Target/proxy used for the leakage test
y = (df["trend_direction"] == "down").astype(int)

# ---------------------------------------------------------
# 1. Check that forbidden leakage fields are not in X
# ---------------------------------------------------------

forbidden_features = [
    "trend_direction",
    "trend_pct",

    # Direct trend-window inputs
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",

    # 90-day performance fields overlap the trend windows
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",

    # Identifiers
    "content_id",
    "client_id",

    # Existing-system/product metadata not used as model inputs
    "impression_tier",
    "position_tier",
    "provider_used",
    "model_used",
]

leakage_present = [
    col for col in forbidden_features
    if col in X.columns
]

print("Forbidden columns found in feature matrix:")
print(leakage_present)

assert len(leakage_present) == 0, (
    "Leakage detected in the feature matrix!"
)

# ---------------------------------------------------------
# 2. Deliberately add the leaky trend_pct feature
# ---------------------------------------------------------

leaky_prediction = (
    pd.to_numeric(df["trend_pct"], errors="coerce")
    .fillna(0)
    < -20
).astype(int)

leakage_accuracy = (leaky_prediction == y).mean()

print("\nDeliberate leakage test:")
print(f"Accuracy using trend_pct alone: {leakage_accuracy:.3f}")

print("\nExpected result:")
print("A near-perfect score confirms that trend_pct contains the label definition.")

Forbidden columns found in feature matrix:
[]

Deliberate leakage test:
Accuracy using trend_pct alone: 1.000

Expected result:
A near-perfect score confirms that trend_pct contains the label definition.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Answer

| Field/group | Why excluded |
|---|---|
| `trend_direction` | Directly defines the decline target |
| `trend_pct` | Source of `trend_direction`; directly reveals the target |
| `impressions_last_30d` | Direct input to the trend calculation |
| `clicks_last_30d` | Same recent outcome window; excluded for temporal safety |
| `sessions_last_30d` | Same recent outcome window; excluded for temporal safety |
| `impressions_prev_30d` | Direct input to the trend calculation |
| `clicks_prev_30d` | Same trend-comparison window |
| `sessions_prev_30d` | Same trend-comparison window |
| 90-day performance totals | Their window overlaps the trend windows |
| `ctr` | Derived from 90-day impressions and clicks |
| `avg_position` | Search-performance outcome measured over the reporting window |
| `engagement_rate` | Derived from 90-day engagement/session data |
| `scroll_rate` | Derived from 90-day activity data |
| `ai_traffic_pct` | Derived from 90-day traffic data |
| `impression_tier` | Derived from search-impression performance |
| `position_tier` | Derived from average search position |
| `content_id` | Pseudonymous identifier; useful for grouping, not prediction |
| `client_id` | Pseudonymous identifier; reserved for grouped splitting |
| `provider_used` | Metadata about the generation system, not needed for this feature vector |
| `model_used` | Metadata about the generation system, not needed for this feature vector |

The resulting feature vector intentionally favors metadata and content properties over measurements that overlap the outcome definition. This makes the feature set more conservative and reduces the risk of learning the answer from the same window used to define the target.

In [8]:
excluded_groups = {
    "label-derived": [
        "trend_direction",
        "trend_pct",
    ],

    "trend-window inputs": [
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d",
        "impressions_prev_30d",
        "clicks_prev_30d",
        "sessions_prev_30d",
    ],

    "overlapping 90-day performance": [
        "impressions_90d",
        "clicks_90d",
        "pageviews_90d",
        "sessions_90d",
        "users_90d",
        "engaged_sessions_90d",
        "ai_sessions_90d",
        "scroll_events_90d",
        "days_with_impressions",
        "days_with_sessions",
        "ctr",
        "avg_position",
        "engagement_rate",
        "scroll_rate",
        "ai_traffic_pct",
    ],

    "identifiers": [
        "content_id",
        "client_id",
    ],

    "derived/search-performance metadata": [
    "impression_tier",
    "position_tier",
    ],

    "generation metadata": [
        "provider_used",
        "model_used",
    ],
}

excluded_fields = [
    field
    for fields in excluded_groups.values()
    for field in fields
]

print("Excluded field count:", len(excluded_fields))
print()

for group, fields in excluded_groups.items():
    print(f"{group}: {len(fields)} fields")
    print(", ".join(fields))
    print()

# Final safety check
feature_source_fields = numeric_features + categorical_features

overlap = set(feature_source_fields).intersection(excluded_fields)

print("Feature/exclusion overlap:", overlap)

assert len(overlap) == 0, (
    "A field appears in both the feature list and exclusion list."
)

Excluded field count: 29

label-derived: 2 fields
trend_direction, trend_pct

trend-window inputs: 6 fields
impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d

overlapping 90-day performance: 15 fields
impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct

identifiers: 2 fields
content_id, client_id

derived/search-performance metadata: 2 fields
impression_tier, position_tier

generation metadata: 2 fields
provider_used, model_used

Feature/exclusion overlap: set()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.